# AI后厨管家项目

In [2]:
import sys
print(sys.executable)

c:\Users\mrche\Desktop\Thanatos\learn\Py_study\LangChain+RAG\LC\.venv\Scripts\python.exe


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

# 1.定义模型

In [4]:
from langchain.chat_models import init_chat_model

DASHSCOPE_API_KEY = os.getenv('DASHSCOPE_API_KEY')
DASHSCOPE_BASE_URL = os.getenv('DASHSCOPE_BASE_URL')

model = init_chat_model(
  model='qwen3.5-omni-plus',
  model_provider='openai',
  base_url=DASHSCOPE_BASE_URL,
  api_key=DASHSCOPE_API_KEY
)

# 2.定义工具

In [5]:
from langchain.tools import tool
from langchain_tavily import TavilySearch

tavily = TavilySearch(
  max_results=5
)

@tool
def web_search(query: str):
  ''' 在线搜索查询 '''
  return tavily.invoke(query)

# 3.添加记忆管理

In [6]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

connection = sqlite3.connect('resources/checkpoint-cook.db', check_same_thread=False)

checkpointer = SqliteSaver(connection)

checkpointer.setup()

# 4.定义智能体

In [7]:
from langchain.agents import create_agent

system_prompt = '''
  你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作:
  1.识别和评估食材:若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用
  食材清单”。
  2.智能食谱检索:优先调用 web_search工具，以“可用食材清单”为核心关键词，查找可行菜谱。
  3.多维度评估与排序:从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
  4.结构化方案输出:把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

  请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
'''

agent = create_agent(
  model=model,
  tools=[web_search],
  checkpointer=checkpointer,
  system_prompt=system_prompt
)

# 5.测试

In [9]:
from langchain.messages import HumanMessage, AIMessage

config = {'configurable': {'thread_id': 'thread_1'}}

multimessage = HumanMessage([
  {'type':'text','text':'帮我看看能做什么？'},
  {'type':'image','url':'https://inews.gtimg.com/om_bt/OXKPyxDKO0e_cUmPUQmQQKcG539iIL-uj1GKckw5LAmLUAA/1000'}
])

response = agent.invoke({'messages': [multimessage]}, config)

In [ ]:
for msg in response['messages']:
  msg.pretty_print()

# 流式输出

In [ ]:
from langchain.messages import AIMessageChunk


def search_recipes(prompt: str, image: str, thread_id: str):
  try:
    if not image or not image.strip():
      messages = HumanMessage(prompt)
    else:
      messages = HumanMessage([
        {'type': 'text', 'text': prompt},
        {'type': 'image', 'url': image}
      ])

    config = {'configurable': {'thread_id': thread_id}}

    response = agent.stream(
      {'messages': [messages]},
      config,
      stream_mode='messages'
    )

    for chunk,metadata in response:
      if isinstance(chunk, AIMessageChunk) and chunk.content:
        yield chunk.content
  except Exception as e:
    print(e)
    yield f'信息检索失败，试试看手动输入食物列表？'

# 获取会话历史

In [10]:
checkpointer.get(config)

{'v': 4,
 'ts': '2026-11-20T08:52:25.594508+00:00',
 'id': '1f1e4bfb-53fe-6252-801a-ab1baa586153',
 'channel_values': {'messages': [HumanMessage(content=[{'type': 'text', 'text': '帮我看看能做什么？'}, {'type': 'image', 'url': 'https://inews.gtimg.com/om_bt/OXKPyxDKO0e_cUmPUQmQQKcG539iIL-uj1GKckw5LAmLUAA/1000'}], additional_kwargs={}, response_metadata={}, id='9096b7b3-da18-45d1-ad32-b1d445f89863'),
   AIMessage(content='好的，收到您的食材照片了。作为一名私人厨师，我将立即为您分析食材并规划美食方案。\n\n首先，我需要识别和评估您冰箱里的所有食材。\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 1183, 'total_tokens': 1262, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': None, 'rejected_prediction_tokens': None, 'text_tokens': 79}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': None, 'image_tokens': 715, 'text_tokens': 468}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-omni-pl

In [ ]:
def get_message(thread_id: str) -> list[dict[str, str]]:
  cp = checkpointer.get(config)
  if not cp:
    return []

  channel_values = cp.get('channel_values')
  if not channel_values:
    return []

  messages = channel_values.get('messages', [])
  if not messages:
    return []

  # 转换消息格式
  result = []
  for msg in messages:
    print(msg)
    if msg.content:
      if isinstance(msg, HumanMessage):
        result.append({'role': 'user', 'content': msg.content})
      elif isinstance(msg, AIMessage):
        result.append({'role': 'assistant', 'content': msg.content})

  return result

# 清空会话历史

In [ ]:
def clear_messsages(thread_id: str):
  checkpoint.delete_thread(thread_id)